In [2]:
import numpy as np
import tensorflow as tf
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings
import random
warnings.filterwarnings('ignore')

In [3]:
seed = 42
np.random.seed(seed)
random.seed(seed)
tf.random.set_seed(seed)

In [4]:
df = pd.read_csv("Air Traffic Data Cor Updated.csv",parse_dates=['Date'],index_col='Date')
df.head()

,domestic passengers,international passenegrs,domestic freight(in tonne),international freight(in tonne),GDP (in dollars),Jet Fuel Price per Gallon,Inflation Rate,Unemployement Rate
Date,,,,,,,,
2009-01-01,3288004,885435,20832,11675,1.341888e+12,71.75,10.88,7.66
2009-02-01,3293220,757168,18645,12482,1.341888e+12,61.97,10.88,7.66
2009-03-01,3122400,848046,23046,15359,1.341888e+12,65.01,10.88,7.66
2009-04-01,3266686,861715,21623,14512,1.341888e+12,68.55,10.88,7.66
2009-05-01,3883887,898410,19534,14586,1.341888e+12,72.22,10.88,7.66


In [5]:
target = ['domestic freight(in tonne)']
exog = ['GDP (in dollars)', 'Jet Fuel Price per Gallon',
        'Inflation Rate ', 'Unemployement Rate']

features = target + exog

In [6]:
# Scale exogenous features
scaler_exog = MinMaxScaler()
scaled_exog = scaler_exog.fit_transform(df[exog])

# Scale targets
scaler_targets = MinMaxScaler()
scaled_targets = scaler_targets.fit_transform(df[target])

# Recombine
scaled_df = pd.DataFrame(
    np.hstack([scaled_targets, scaled_exog]),
    index=df.index,
    columns=target + exog
)

In [7]:

def create_sequences(data, target_col, seq_length=12):
    X, y = [], []
    target_idx = data.columns.get_loc(target_col)
    values = data.values
    
    for i in range(len(values) - seq_length):
        X.append(values[i:i+seq_length])        # input: seq_length months of all features
        y.append(values[i+seq_length, target_idx])  # output: next-step target
    
    return np.array(X), np.array(y)

SEQ_LEN = 12
X, y = create_sequences(scaled_df, 'domestic passengers', SEQ_LEN)

print("X shape:", X.shape)   # (samples, 12, num_features)
print("y shape:", y.shape)   # (samples,)

KeyError: 'domestic passengers'

In [ ]:
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

In [ ]:
# Build GRU model
model_gru = Sequential()
model_gru.add(GRU(32, activation='tanh',return_sequences=True, input_shape=(SEQ_LEN, len(features))))
model_gru.add(Dropout(0.2))
model_gru.add(GRU(16,activation='tanh'))
model_gru.add(Dense(1))   # predict only Domestic Passengers
model_gru.compile(optimizer='RMSprop', loss='mse')
model_gru.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 12, 32)         │         3,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 12, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 16)             │         2,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,161 (24.07 KB)

 Trainable params: 6,161 (24.07 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [ ]:
# Train GRU
history_gru = model_gru.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    # callbacks=[early_stop],
    batch_size=32,
    verbose=1
)

Epoch 1/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 136ms/step - loss: 0.0323 - val_loss: 0.0853
Epoch 2/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 0.0199 - val_loss: 0.0692
Epoch 3/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 0.0201 - val_loss: 0.0714
Epoch 4/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.0191 - val_loss: 0.0675
Epoch 5/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.0171 - val_loss: 0.0647
Epoch 6/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.0152 - val_loss: 0.0682
Epoch 7/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.0163 - val_loss: 0.0526
Epoch 8/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.0138 - val_loss: 0.0481
Epoch 9/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.0151 - val_loss: 0.0462
Epoch 10/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.0153 - val_loss: 0.0450
Epoch 11/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.0136 - val_loss: 0.0444
Epoch 12/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 0.0142 - val_loss: 0.0516


In [ ]:
y_pred_gru = model_gru.predict(X_test)

# Inverse scaling
y_test_inv = scaler_targets.inverse_transform(y_test.reshape(-1,1))
y_pred_inv = scaler_targets.inverse_transform(y_pred_gru)

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 375ms/step


In [ ]:
# Evaluation
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score , mean_absolute_percentage_error 
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
mae = mean_absolute_error(y_test_inv, y_pred_inv)
r2 = r2_score(y_test_inv, y_pred_inv)
mape = mean_absolute_percentage_error(y_test_inv, y_pred_inv)

print(f"GRU - Domestic Passengers → RMSE={rmse:.2f}, MAE={mae:.2f},MAPE={mape:.2f}, R²={r2:.3f}")

GRU - Domestic Passengers → RMSE=2032529.22, MAE=1891578.43,MAPE=0.16, R²=-0.408


In [8]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))
plt.plot(y_test_inv, label="Actual")
plt.plot(y_pred_inv, label="GRU Predicted")
plt.title("Domestic Passengers - GRU Actual vs Predicted")
plt.legend()
plt.show()


NameError: name 'y_test_inv' is not defined

<Figure size 1000x400 with 0 Axes>

In [9]:
# Naïve baseline: predict next value as last observed value
y_naive = y_test_inv[:-1]  
baseline_rmse = np.sqrt(mean_squared_error(y_test_inv[1:], y_naive))
baseline_mae = mean_absolute_error(y_test_inv[1:], y_naive)

print(f"Naïve Baseline → RMSE={baseline_rmse:.2f}, MAE={baseline_mae:.2f}")


NameError: name 'y_test_inv' is not defined